# Experimenting with Training Techniques (Gradient Penalties, Spectral Normalization)

## 📚 Learning Objectives

By completing this notebook, you will:
- Implement gradient penalties
- Implement spectral normalization
- Stabilize GAN training
- Improve training stability
- Apply advanced training techniques

## 🔗 Prerequisites

- ✅ Understanding of GAN training
- ✅ Understanding of normalization
- ✅ TensorFlow/PyTorch knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 1**:
- Experimenting with training techniques like gradient penalties and spectral normalization
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 1 Practical Content

---

## Introduction

**Advanced training techniques** like gradient penalties and spectral normalization improve GAN training stability and convergence, addressing common challenges in generative model training.

## 📥 Inputs & 📤 Outputs | المدخلات والمخرجات

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [ ]:
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
print(f'PyTorch {torch.__version__}')
print('✅ Libraries imported!')
print('\nTraining Techniques: Gradient Penalties and Spectral Normalization')
print('=' * 60)

# Experimenting with Training Techniques (Gradient Penalties, Spectral Normalization)

## 📚 Learning Objectives

By completing this notebook, you will:
- Apply gradient penalties
- Use spectral normalization
- Improve GAN training stability
- Experiment with techniques
- Evaluate improvements

## 🔗 Prerequisites

- ✅ Understanding of GAN training
- ✅ Understanding of regularization
- ✅ TensorFlow/PyTorch knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 10, Unit 1**:
- Experimenting with training techniques like gradient penalties and spectral normalization
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md` - Unit 1 Practical Content

---

## Introduction

**Training techniques** like gradient penalties and spectral normalization improve GAN training stability and sample quality.

In [ ]:
# ── Synthetic 2-D data ───────────────────────────────────────────────────────
torch.manual_seed(0)
real_data = torch.randn(800, 2)
latent_dim, batch_size = 8, 64

Generator     = nn.Sequential(nn.Linear(latent_dim, 32), nn.ReLU(), nn.Linear(32, 2))
# Discriminator with Spectral Normalization on each Linear layer
Discriminator = nn.Sequential(
    nn.utils.spectral_norm(nn.Linear(2, 32)), nn.LeakyReLU(0.2),
    nn.utils.spectral_norm(nn.Linear(32, 1)),  # raw score (WGAN-style)
)

opt_G = optim.RMSprop(Generator.parameters(),     lr=5e-4)
opt_D = optim.RMSprop(Discriminator.parameters(), lr=5e-4)

def gradient_penalty(D, real, fake, lam=10.0):
    """WGAN-GP: penalise the gradient norm of D interpolated between real and fake."""
    alpha  = torch.rand(real.size(0), 1)
    interp = (alpha * real + (1 - alpha) * fake).requires_grad_(True)
    score  = D(interp)
    grads  = torch.autograd.grad(score, interp,
                                  grad_outputs=torch.ones_like(score),
                                  create_graph=True)[0]
    return lam * ((grads.norm(2, dim=1) - 1) ** 2).mean()

g_losses, d_losses = [], []
for epoch in range(200):
    # Train D (5 steps per G step — standard WGAN schedule)
    for _ in range(5):
        idx  = torch.randint(0, len(real_data), (batch_size,))
        real = real_data[idx]
        fake = Generator(torch.randn(batch_size, latent_dim)).detach()
        gp   = gradient_penalty(Discriminator, real, fake)
        loss_D = -Discriminator(real).mean() + Discriminator(fake).mean() + gp
        opt_D.zero_grad(); loss_D.backward(); opt_D.step()
    # Train G
    fake   = Generator(torch.randn(batch_size, latent_dim))
    loss_G = -Discriminator(fake).mean()
    opt_G.zero_grad(); loss_G.backward(); opt_G.step()
    g_losses.append(loss_G.item()); d_losses.append(loss_D.item())

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 3))
plt.plot(g_losses[::10], label='G'); plt.plot(d_losses[::10], label='D')
plt.title('WGAN-GP Training Loss'); plt.legend(); plt.tight_layout(); plt.show()
print('Key points demonstrated:')
print('  • Spectral Normalization: applied via nn.utils.spectral_norm()')
print('  • Gradient Penalty (WGAN-GP): computed with torch.autograd.grad()')
print('  • These stabilize GAN training compared to vanilla BCE loss.')

## 🌍 Real-World Worked Example — House Price Prediction (Manual Backprop)

**Industry context:** Real estate platforms (Zillow, Property Finder) use regression models  
trained with gradient descent to estimate property prices.

Below we implement a 1-hidden-layer network **with manual backpropagation** to predict house prices.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

np.random.seed(42)
# ── Synthetic house data (size in m², price in 100k SAR) ──────────────────
n = 200
X = np.random.uniform(50, 300, (n, 1)).astype(np.float32)       # size
y = (2.5 * X + np.random.randn(n,1)*30).astype(np.float32)      # price

# Normalise
X_n = (X - X.mean()) / X.std()
y_n = (y - y.mean()) / y.std()

# ── Manual 1-layer network ─────────────────────────────────────────────────
def relu(z): return np.maximum(0, z)
def relu_d(z): return (z > 0).astype(float)

W1 = np.random.randn(1, 16) * 0.1;  b1 = np.zeros((1, 16))
W2 = np.random.randn(16, 1) * 0.1;  b2 = np.zeros((1, 1))
lr = 0.01;  losses = []

for _ in range(500):
    # Forward
    z1 = X_n @ W1 + b1          # (n, 16)
    a1 = relu(z1)
    z2 = a1 @ W2 + b2            # (n, 1)
    loss = ((z2 - y_n)**2).mean()
    losses.append(loss)

    # ── Backpropagation (chain rule, step by step) ─────────────────────────
    dL_dz2 = 2*(z2 - y_n) / n
    dL_dW2 = a1.T @ dL_dz2
    dL_db2 = dL_dz2.sum(0, keepdims=True)
    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * relu_d(z1)
    dL_dW1 = X_n.T @ dL_dz1
    dL_db1 = dL_dz1.sum(0, keepdims=True)

    # Gradient descent step
    W2 -= lr * dL_dW2;  b2 -= lr * dL_db2
    W1 -= lr * dL_dW1;  b1 -= lr * dL_db1

print(f"Final MSE loss: {losses[-1]:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(losses); plt.title("Loss Curve — House Price Model"); plt.xlabel("Epoch")
plt.subplot(1, 2, 2)
pred_y = (relu(X_n @ W1 + b1) @ W2 + b2) * y.std() + y.mean()
plt.scatter(X, y, alpha=0.3, label="Real data")
idx = X[:,0].argsort()
plt.plot(X[idx], pred_y[idx], 'r-', lw=2, label="Model prediction")
plt.xlabel("House size (m²)"); plt.ylabel("Price (100k SAR)"); plt.legend()
plt.title("Real-World: House Price Prediction")
plt.tight_layout(); plt.show()

## 📚 References & Further Reading

**Papers:**
- Rumelhart, Hinton & Williams (1986) — [Learning representations by back-propagating errors](https://www.nature.com/articles/323533a0)

**Tutorials:**
- [Andrej Karpathy — micrograd](https://github.com/karpathy/micrograd) (build backprop from scratch in 150 lines)
- [PyTorch Autograd Tutorial](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html)

**State-of-the-Art:** Backpropagation is the engine behind GPT-4, Stable Diffusion, and every modern neural network.

## 📝 Summary

In this notebook you studied **07 Training Techniques Gradient Penalties** — a key component of modern AI systems. The concepts covered here connect directly to production systems used by leading tech companies. Review the examples, experiment with the code, and check the references for deeper study.